# RAG (검색 증강 생성) : FAISS 사용
: https://www.promptingguide.ai/kr/techniques/rag

문서를 벡터로 변환(임베딩) : 텍스트 데이터를 고차원 벡터 공간으로 매핑하는 작업

### FAISS (Facebook AI Similarity Search)
Facebook AI 팀에서 개발한 대규모 벡터 검색 라이브러리로, 빠르고 효율적인 최근접 이웃 검색(Nearest Neighbor Search, NNS) 을 수행하는 데 사용된다. 
<br>
특히, 고차원 벡터 검색을 최적화하여 딥러닝 임베딩 검색, 추천 시스템, RAG(Retrieval-Augmented Generation) 등에서 널리 활용된다

In [1]:
from dotenv import load_dotenv
import os

# .env 파일 불러오기
load_dotenv("C:/env/.env")

# 환경 변수 가져오기
API_KEY = os.getenv("OPENAI_API_KEY")

from openai import OpenAI
client = OpenAI(api_key=API_KEY)

In [3]:
# ! pip install faiss-cpu
import faiss
import numpy as np

# 샘플 문서데이터 (텍스트)
documents = [
    "OpenAI는 인공지능 연구소입니다.",
    "FAISS는 Facebook AI에서 개발한 벡터 검색 라이브러리입니다.",
    "GPT는 자연어 처리를 위한 강력한 AI 모델입니다.",
    "RAG는 검색 기반 생성 기법을 활용하여 AI 응답을 향상시키는 방법입니다."    
]

In [4]:
# 문서를 벡터로 변환 (임베딩)
# https://platform.openai.com/docs/pricing
def get_embedding(text):
    response = client.embeddings.create(
        model = 'text-embedding-3-small',   # text-embedding-ada-002, text-embedding-3-large
        input = text        
    )
    return np.array(response.data[0].embedding)

# 모든 문서의 임베딩 생성
document_embeddings = np.array([get_embedding(doc) for doc in documents ])
print(document_embeddings)

[[-0.00731277 -0.00444031  0.01930237 ...  0.00988007 -0.00136375
   0.00521851]
 [-0.01360321  0.00777435 -0.02600098 ...  0.02081299 -0.03369141
  -0.0146637 ]
 [-0.01107025  0.07653809  0.03829956 ... -0.02166748 -0.00357437
   0.00450134]
 [ 0.01522064  0.02207947  0.00048733 ... -0.01212311 -0.01119232
  -0.03970337]]


In [5]:
print(document_embeddings[0])        # "OpenAI는 인공지능 연구소입니다."
print(len(document_embeddings[0]))   # 1536

[-0.00731277 -0.00444031  0.01930237 ...  0.00988007 -0.00136375
  0.00521851]
1536


In [6]:
# FAISS 벡터 인덱스 생성
dimension = document_embeddings.shape[1]   
print(dimension)    # 1536
index = faiss.IndexFlatL2(dimension)
index.add(document_embeddings)

1536


In [7]:
#  사용자 질문 입력 및 벡터 변환
query = "RAG의 핵심 구성 요소는 무엇인가?"
query_embedding = get_embedding(query)
print(query_embedding)

[ 0.015625    0.00235367 -0.03765869 ... -0.00117397 -0.01971436
 -0.01447296]


In [8]:
#  FAISS를 이용한 검색( 가장 유사한 문서 찾기)
k = 2  # 가장 유사한 2개의 문서 검색
distances,indices = index.search(query_embedding.reshape(1,-1),k)
print(distances, indices)

[[1.0508488 1.5755229]] [[3 1]]


In [8]:
# # 배열의 reshape
# a = np.arange(12)
# print(a.shape)  # 1차원 배열
# print(a)
# a2 = a.reshape(3,4)
# print(a2.shape)  # 2차원 배열
# print(a2)
# a3 = a.reshape(2,3,2) 
# print(a3.shape)  # 3차원 배열
# print(a3)

# b = a.reshape(3,-1)  # -1은 shape을 자동으로 계산
# print(b,b.shape)

In [9]:
# 검색된 문서 내용 추출
retrieved_docs = [documents[i] for i in indices[0]]
print(retrieved_docs)

['RAG는 검색 기반 생성 기법을 활용하여 AI 응답을 향상시키는 방법입니다.', 'FAISS는 Facebook AI에서 개발한 벡터 검색 라이브러리입니다.']


In [10]:
# LLM을 활용하여 최종 응답 생성
context = "\n".join(retrieved_docs)
prompt = f"질문: {query}\n\n참고자료:\n{context}\n\n답변:"
print(prompt)

completion = client.chat.completions.create(
    model = "gpt-4o-mini", 
    messages = [{"role": "system", "content": "당신은 유용한 AI 어시스턴트입니다."},
                {"role":"user","content":prompt} ]
)    

final_answer = completion.choices[0].message.content
print("🔹 AI 응답:", final_answer)

질문: RAG의 핵심 구성 요소는 무엇인가?

참고자료:
RAG는 검색 기반 생성 기법을 활용하여 AI 응답을 향상시키는 방법입니다.
FAISS는 Facebook AI에서 개발한 벡터 검색 라이브러리입니다.

답변:
🔹 AI 응답: RAG(지식 기반 생성, Retrieval-Augmented Generation)의 핵심 구성 요소는 다음과 같습니다:

1. **문서 검색기 (Retriever)**: 이 구성 요소는 주어진 질문이나 문맥에 대해 관련 문서를 검색합니다. 일반적으로 FAISS와 같은 벡터 검색 라이브러리를 사용하여 대규모 데이터 세트에서 최적의 문서를 신속하게 찾습니다.

2. **생성기 (Generator)**: 검색된 문서를 기반으로 최종 응답을 생성하는 모델입니다. 일반적으로 Transformer 모델(예: BERT, GPT)를 사용하여 자연어를 생성합니다.

3. **데이터베이스 (Knowledge Base)**: RAG는 외부 지식 데이터베이스에 접근하여 필요한 정보를 얻습니다. 이 데이터베이스는 도메인 특정 데이터, 매뉴얼, 위키피디아 문서와 같은 다양한 출처의 정보를 포함할 수 있습니다.

이 세 가지 구성 요소가 함께 작동하여 RAG는 보다 정확하고 관련성 높은 응답을 생성하는 데 기여합니다.
